In [1]:
import torch
from matplotlib import pyplot as plt
from tools.embs_tools import get_embs, aggregate_embeddings, accelerated_cosine_similarity, bb_weighted_average
import numpy as np
import itertools

In [2]:
# all_embs = get_embs("/home/simon/car_outputs/")
# avg_fn = lambda embs: np.mean(embs, axis=0)
# all_embs_aggregated = {path: aggregate_embeddings(file_embs, aggregation_fn=avg_fn) for path, file_embs in all_embs.items()}

In [3]:
all_embs = get_embs("/home/simon/new_outputs/")
all_embs_aggregated = {
    path: aggregate_embeddings(file_embs, aggregation_fn=bb_weighted_average)
    for path, file_embs in all_embs.items()
}

ERROR:root:No meta file found in /home/simon/new_outputs/SD-PO-P576__20250519-062004/2025_07/12/19/24/track_meta.csv, skipping...
Aggregating embeddings: 100%|██████████| 171494/171494 [00:00<00:00, 5998548.69it/s]


In [4]:
embs = [torch.stack([torch.tensor(e) for _, e in embs]) for _, embs in all_embs_aggregated.items()]
embs = torch.stack(list(itertools.chain.from_iterable(embs)))
similarity = accelerated_cosine_similarity(embs, embs, batch_size=2048)
similarity = similarity.numpy()
similarity = similarity[np.triu_indices(similarity.shape[0], k=1)].flatten()

plt.hist(similarity, bins=100, range=(-1, 1), density=True)
plt.xlim([-0.5, 1.0])
plt.ylabel('Number of samples [mil]')
plt.xlabel('Cosine similarity score')
plt.title('Histogram of similarity scores')

/tmp/ipykernel_571400/1092205643.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  embs = [torch.stack([torch.tensor(e) for _, e in embs]) for _, embs in all_embs_aggregated.items()]


RuntimeError: [enforce fail at alloc_cpu.cpp:119] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 1016495469796 bytes. Error code 12 (Cannot allocate memory)